In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from satelliteSegmentation.tokenizer import IDX2LABEL, NICECOLOR
from satelliteSegmentation.utils import plot_confusion_matrix

In [ ]:
def plot_compare(hist: dict[str, pd.DataFrame]):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax = ax.flatten()

    for name, df in hist.items():
        ax[0].plot(range(1, len(df) + 1), df["val_loss"], label=name)

    mins = pd.DataFrame(
        [[name, min(hist[name].val_loss)] for name in hist.keys()],
        columns=["model", "val_loss"],
    )

    sns.barplot(
        mins,
        y="model",
        x="val_loss",
        orient="h",
        order=mins.sort_values("val_loss")["model"],
        ax=ax[1],
    )

    # Add value to each bar
    for bar in ax[1].patches:
        value = bar.get_width()
        ax[1].text(
            value,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.4f}",
            va="center",
            ha="right",
            fontsize=10,
        )

    ax[0].legend()
    plt.tight_layout()
    plt.show()


def open_hist(folder: Path | str):
    folder = folder if isinstance(folder, Path) else Path(folder)

    hist: dict[str, pd.DataFrame] = {}
    for model_folder in folder.glob("*"):
        if model_folder.is_file():
            raise RuntimeError(
                f"Found a file while searching for model folders. Folder at: {model_folder.resolve().as_posix()}"
            )

        if not model_folder.name.startswith("_"):
            hist[model_folder.name] = pd.read_csv(model_folder / "hist.csv")

    return hist

## Comparar Modelos

In [ ]:
hist_lr = open_hist("../results/att-unet/lr")

In [ ]:
plot_compare(hist_lr)

In [ ]:
ax = sns.pointplot(
    pd.DataFrame([[float(k.replace("lr-", "").replace("baseline", "1e-4")) , v["val_loss"].min()] for k, v in hist_lr.items()], columns=["lr", "val_loss"]),
    x="lr", y="val_loss",
);
plt.xticks(rotation=45);

In [ ]:
hist_k = open_hist("../results/att-unet/k")

In [ ]:
plot_compare(hist_k)

In [ ]:
hist_wd = open_hist("../results/att-unet/wd")

In [ ]:
plot_compare(hist_wd)

## Cross Validation